In [1]:
import os, sys
import json
import torch

# defer loading `lm_eval` submodules for faster CLI load
from lm_eval import evaluator, utils
from lm_eval.evaluator import request_caching_arg_to_dict
from lm_eval.loggers import EvaluationTracker, WandbLogger
from lm_eval.tasks import TaskManager
from lm_eval.utils import (
    handle_non_serializable,
    make_table,
    simple_parse_args_string,
)
from lm_eval.models.huggingface import HFLM

/home/kimth/tools/miniconda3/envs/lm_eval/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
if 0:
    if args.predict_only:
        args.log_samples = True
    if (args.log_samples or args.predict_only) and not args.output_path:
        raise ValueError(
            "Specify --output_path if providing --log_samples or --predict_only"
        )

    if args.fewshot_as_multiturn and args.apply_chat_template is False:
        raise ValueError(
            "When `fewshot_as_multiturn` is selected, `apply_chat_template` must be set (either to `True` or to the chosen template name)."
        )

    if args.include_path is not None:
        eval_logger.info(f"Including path: {args.include_path}")
    metadata = (
        simple_parse_args_string(args.model_args)
        if isinstance(args.model_args, str)
        else args.model_args
        if isinstance(args.model_args, dict)
        else {}
    ) | (
        args.metadata
        if isinstance(args.metadata, dict)
        else simple_parse_args_string(args.metadata)
    )

    task_manager = TaskManager(include_path=args.include_path, metadata=metadata)

    if "push_samples_to_hub" in evaluation_tracker_args and not args.log_samples:
        eval_logger.warning(
            "Pushing samples to the Hub requires --log_samples to be set. Samples will not be pushed to the Hub."
        )

    if args.limit:
        eval_logger.warning(
            " --limit SHOULD ONLY BE USED FOR TESTING."
            "REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT."
        )
    if args.samples:
        assert args.limit is None, (
            "If --samples is not None, then --limit must be None."
        )
        if (samples := Path(args.samples)).is_file():
            args.samples = json.loads(samples.read_text())
        else:
            args.samples = json.loads(args.samples)

    if args.tasks is None:
        eval_logger.error("Need to specify task to evaluate.")
        sys.exit()
    elif args.tasks == "list":
        print(task_manager.list_all_tasks())
        sys.exit()
    elif args.tasks == "list_groups":
        print(task_manager.list_all_tasks(list_subtasks=False, list_tags=False))
        sys.exit()
    elif args.tasks == "list_tags":
        print(task_manager.list_all_tasks(list_groups=False, list_subtasks=False))
        sys.exit()
    elif args.tasks == "list_subtasks":
        print(task_manager.list_all_tasks(list_groups=False, list_tags=False))
        sys.exit()
    else:
        if os.path.isdir(args.tasks):
            import glob

            task_names = []
            yaml_path = os.path.join(args.tasks, "*.yaml")
            for yaml_file in glob.glob(yaml_path):
                config = utils.load_yaml_config(yaml_file)
                task_names.append(config)
        else:
            task_list = args.tasks.split(",")
            task_names = task_manager.match_tasks(task_list)
            for task in [task for task in task_list if task not in task_names]:
                if os.path.isfile(task):
                    config = utils.load_yaml_config(task)
                    task_names.append(config)
            task_missing = [
                task for task in task_list if task not in task_names and "*" not in task
            ]  # we don't want errors if a wildcard ("*") task name was used

            if task_missing:
                missing = ", ".join(task_missing)
                eval_logger.error(
                    f"Tasks were not found: {missing}\n"
                    f"{utils.SPACING}Try `lm-eval --tasks list` for list of available tasks",
                )
                raise ValueError(
                    f"Tasks not found: {missing}. Try `lm-eval --tasks {{list_groups,list_subtasks,list_tags,list}}` to list out all available names for task groupings; only (sub)tasks; tags; or all of the above, or pass '--verbosity DEBUG' to troubleshoot task registration issues."
                )

    # Respect user's value passed in via CLI, otherwise default to True and add to comma-separated model args
    if args.trust_remote_code:
        eval_logger.info(
            "Passed `--trust_remote_code`, setting environment variable `HF_DATASETS_TRUST_REMOTE_CODE=true`"
        )
        # HACK: import datasets and override its HF_DATASETS_TRUST_REMOTE_CODE value internally,
        # because it's already been determined based on the prior env var before launching our
        # script--`datasets` gets imported by lm_eval internally before these lines can update the env.
        import datasets
        from packaging.version import parse as vparse

        if vparse(datasets.__version__) < vparse("4.0.0"):
            datasets.config.HF_DATASETS_TRUST_REMOTE_CODE = True

        if isinstance(args.model_args, dict):
            args.model_args["trust_remote_code"] = True
        else:
            args.model_args = args.model_args + ",trust_remote_code=True"
    (
        eval_logger.info(f"Selected Tasks: {task_names}")
        if eval_logger.getEffectiveLevel() >= logging.INFO
        else print(f"Selected Tasks: {task_names}")
    )

    request_caching_args = request_caching_arg_to_dict(
        cache_requests=args.cache_requests
    )

In [3]:
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM

STORAGE_PATH="/DATA2/kimth/"
MODEL_PATH=f"{STORAGE_PATH}/models"

model_id = f"{MODEL_PATH}/Mixtral-8x7B-Instruct-v0.1"
# model_id = f"{MODEL_PATH}/Phi-3.5-MoE-instruct"
# model_id=f"{MODEL_PATH}/OLMo-1B-0724-hf"

if "mixtral" in model_id.lower() :
    from my_models.modeling_mixtral_skip import MixtralForCausalLM, set_gate_threshold, get_n_skipped
    model_cls = MixtralForCausalLM
    model_name = "mixtral"
elif "phi" in model_id.lower() :
    from my_models.modeling_phimoe_skip import PhiMoEForCausalLM, set_gate_threshold, get_n_skipped
    model_cls = PhiMoEForCausalLM
    model_name = "phi"
else:
    model_cls = AutoModelForCausalLM
    model_name = "temp"
    # assert 0

config = AutoConfig.from_pretrained(
    model_id,
    trust_remote_code=True,
)
# setattr(config, "_attn_implementation", "eager")
setattr(config, "_attn_implementation", "sdpa")

# debug
setattr(config, "num_hidden_layers", 1)

my_model = model_cls.from_pretrained(
    model_id,
    config=config,
    trust_remote_code=True,
    # torch_dtype=torch.bfloat16,
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    # ignore_mismatched_sizes=True,
    # device_map=device_map
    device_map="cpu",
    # device_map="auto",
)

Loading checkpoint shards: 100%|██████████| 19/19 [00:00<00:00, 1057.76it/s]
Some weights of the model checkpoint at /DATA2/kimth//models/Mixtral-8x7B-Instruct-v0.1 were not used when initializing MixtralForCausalLM: ['model.layers.1.block_sparse_moe.experts.0.w1.weight', 'model.layers.1.block_sparse_moe.experts.0.w2.weight', 'model.layers.1.block_sparse_moe.experts.0.w3.weight', 'model.layers.1.block_sparse_moe.experts.1.w1.weight', 'model.layers.1.block_sparse_moe.experts.1.w2.weight', 'model.layers.1.block_sparse_moe.experts.1.w3.weight', 'model.layers.1.block_sparse_moe.experts.2.w1.weight', 'model.layers.1.block_sparse_moe.experts.2.w2.weight', 'model.layers.1.block_sparse_moe.experts.2.w3.weight', 'model.layers.1.block_sparse_moe.experts.3.w1.weight', 'model.layers.1.block_sparse_moe.experts.3.w2.weight', 'model.layers.1.block_sparse_moe.experts.3.w3.weight', 'model.layers.1.block_sparse_moe.experts.4.w1.weight', 'model.layers.1.block_sparse_moe.experts.4.w2.weight', 'model.layer

In [4]:
# get args

n_iters = 16
n_samples = 3
batch_size = 3

if 1:
    model                       = HFLM(pretrained=my_model)
    model_args                  = None
    num_fewshot                 = None
    max_batch_size              = None
    device                      = "cuda"
    use_cache                   = None
    limit                       = n_samples
    samples                     = None
    check_integrity             = None
    write_out                   = True
    log_samples                 = True
    evaluation_tracker          = None
    system_instruction          = None
    fewshot_as_multiturn        = False
    apply_chat_template         = fewshot_as_multiturn
    gen_kwargs                  = {"max_gen_toks": n_iters}
    include_path                = None
    metadata                    = None
    task_manager                = TaskManager(include_path=include_path, metadata=metadata)
    predict_only                = False
    random_seed                 = 0
    numpy_random_seed           = 1234
    torch_random_seed           = 1234
    fewshot_random_seed         = 1234
    confirm_run_unsafe_code     = True
    show_config                 = False


`pretrained` model kwarg is not of type `str`. Many other model arguments may be ignored. Please do not launch via accelerate or use `parallelize=True` if passing an existing model this way.
Passed an already-initialized model through `pretrained`, assuming single-process call to evaluate() or custom distributed integration


In [5]:
# resolve tasks
task_list=[
    # "hendrycks_math_algebra",
    "gsm8k",
    # "gsm8k_llama",
    # "gsm8k_cot",
    # "gsm8k_cot_llama",
]
tasks = ",".join(task_list)

if 1:
    if os.path.isdir(tasks):
        import glob

        task_names = []
        yaml_path = os.path.join(tasks, "*.yaml")
        for yaml_file in glob.glob(yaml_path):
            config = utils.load_yaml_config(yaml_file)
            task_names.append(config)
    else:
        task_list = tasks.split(",")
        task_names = task_manager.match_tasks(task_list)
        for task in [task for task in task_list if task not in task_names]:
            if os.path.isfile(task):
                config = utils.load_yaml_config(task)
                task_names.append(config)
        task_missing = [
            task for task in task_list if task not in task_names and "*" not in task
        ]  # we don't want errors if a wildcard ("*") task name was used

        if task_missing:
            missing = ", ".join(task_missing)
            # eval_logger.error(
            print(
                f"Tasks were not found: {missing}\n"
                f"{utils.SPACING}Try `lm-eval --tasks list` for list of available tasks",
            )
            raise ValueError(
                f"Tasks not found: {missing}. Try `lm-eval --tasks {{list_groups,list_subtasks,list_tags,list}}` to list out all available names for task groupings; only (sub)tasks; tags; or all of the above, or pass '--verbosity DEBUG' to troubleshoot task registration issues."
            )

In [6]:
for gate_threshold in [
    0,
    # 0.1,
    # 0.2,
    # 0.3,
    # 0.4,
    # 0.45,
    0.49,
]:
    set_gate_threshold(gate_threshold)

    results = evaluator.simple_evaluate(
        model                   =model,
        model_args              =model_args,
        tasks                   =task_names,
        num_fewshot             =num_fewshot,
        batch_size              =batch_size,
        max_batch_size          =max_batch_size,
        device                  =device,
        use_cache               =use_cache,
        limit                   =limit,
        samples                 =samples,
        check_integrity         =check_integrity,
        write_out               =write_out,
        log_samples             =log_samples,
        evaluation_tracker      =evaluation_tracker,
        system_instruction      =system_instruction,
        apply_chat_template     =apply_chat_template,
        fewshot_as_multiturn    =fewshot_as_multiturn,
        gen_kwargs              =gen_kwargs,
        task_manager            =task_manager,
        predict_only            =predict_only,
        random_seed             =random_seed,
        numpy_random_seed       =numpy_random_seed,
        torch_random_seed       =torch_random_seed,
        fewshot_random_seed     =fewshot_random_seed,
        confirm_run_unsafe_code =confirm_run_unsafe_code,
        metadata                =metadata,
        # **request_caching_args,
    )

    if hasattr(config, "num_experts_per_tok"):
        total_expert_ops = config.num_hidden_layers * config.num_experts_per_tok * n_iters * n_samples
        print("Thresh={}, Skipped experts {} / {}".format(gate_threshold, get_n_skipped(), total_expert_ops))
    else:
        print("No skipping")

generation_kwargs: {'max_gen_toks': 16} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!


Running generate_until requests: 100%|██████████| 3/3 [00:12<00:00,  4.07s/it]
generation_kwargs: {'max_gen_toks': 16} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!


Thresh=0, Skipped experts 0 / 96


Running generate_until requests: 100%|██████████| 3/3 [00:11<00:00,  3.84s/it]


Thresh=0.49, Skipped experts 41 / 96
